# AutoHetero — showcase

**AutoHetero runs each PyTorch call on the CPU, the GPU, or both — whichever is predicted fastest.**

Run the cells top to bottom on the machine with the NVIDIA GPU. Without one, every step still runs, on the CPU only, and the load step is skipped.

```bash
pip install -e ".[demo]" jupyter
jupyter notebook demo/showcase.ipynb
```

| step | shows |
|---|---|
| 1 | the machine, and the one line that turns AutoHetero on |
| 2 | the problem: the GPU isn't always faster |
| 3 | live: the same op, small and large, and where AutoHetero runs each |
| 4 | live: your own function, numpy in and numpy out |
| 5 | live: a second job on the GPU, and AutoHetero rebalancing |
| 6 | what ran where: `autohetero.report()` |
| 7 | the measured results from the fixed machine |
| 8 | limitations and what comes next |

## Step 1 — the machine, and the one line

`autohetero.enable()` is the only AutoHetero line a program needs. The first time on a machine it calibrates (about 30 s): it measures how fast data moves to the GPU and back, and times every built-in op on both devices. After that it reads the result from its cache.

In [ ]:
import contextlib
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "autohetero").is_dir() else Path.cwd().parent  # run from the repo or demo/
sys.path.insert(0, str(ROOT))

import autohetero
from autohetero import calibrate, devices, history, ops
from autohetero.ops import inputs

autohetero.enable()  # <- the one line

info = calibrate.calibration()
machine = {"CPU": calibrate.machine()["cpu"], "GPU": calibrate.machine()["gpu"] or "none: everything runs on the CPU"}
if info:
    machine |= {"calibration (once)": f"{info['seconds']} s",
                "CPU -> GPU": f"{info['h2d_bytes_per_s'] / 1e9:.1f} GB/s",
                "GPU -> CPU": f"{info['d2h_bytes_per_s'] / 1e9:.1f} GB/s",
                "starting work on the GPU": f"{info['launch_seconds'] * 1e6:.0f} µs"}
pd.Series(machine, name="this machine").to_frame()

## Step 2 — the problem: the GPU isn't always faster

Measured on the fixed machine (RTX 3050 Laptop + Ryzen 7 5800H): every built-in op at growing sizes, on both devices. Below 1× the CPU wins. For most ops the CPU wins small inputs and the GPU wins large ones; for fft the CPU wins at every size. So "always use the GPU" is wrong, and the right choice depends on the op and the input size.

In [ ]:
display(Image(ROOT / "eval" / "out" / "crossover.png"))
pd.read_csv(ROOT / "eval" / "out" / "crossover.csv")[["op", "cpu_faster_up_to_elements", "gpu_faster_from_elements", "gpu_speedup_min", "gpu_speedup_max"]]

## Step 3 — live: the same op, small and large

`compare(op, size)` runs one built-in op with AutoHetero deciding, then with each option forced, so the decision can be checked against what each option really takes. Each option runs once untimed first, as a warm-up.

In [ ]:
OPTIONS = {"AutoHetero": None, "GPU only": "gpu", "Hybrid": "hybrid", "CPU only": "cpu"}  # CPU last: a GPU run right after a long CPU run is slowed


def run(fn, args, mode):
    with autohetero.force(mode) if mode else contextlib.nullcontext():
        start = time.perf_counter()
        fn(*args)
        return time.perf_counter() - start, history.last()


def ran_on(r):
    return r["mode"].upper() + (" -> HYBRID" if r["slowdown"] else "") + (" (GPU failed)" if r["fallback"] else "")


def compare(op, size):
    options = {k: m for k, m in OPTIONS.items() if devices.GPU is not None or m in (None, "cpu")}
    if op not in ops.SPLITTABLE:
        options.pop("Hybrid", None)  # only splittable calls can be shared between the devices
    fn, args = getattr(ops, op), inputs.make(op, size, torch.Generator().manual_seed(0))
    for mode in options.values():
        run(fn, args, mode)  # warm-up: copying a model, loading GPU kernels, deciding
    results = {name: run(fn, args, mode) for name, mode in options.items()}
    predicted = results["AutoHetero"][1]["predicted"]
    table = pd.DataFrame([(name, s * 1e3, ran_on(r), predicted.get(m or r["mode"], float("nan")) * 1e3)
                          for (name, (s, r)), m in zip(results.items(), options.values())],
                         columns=["option", "measured ms", "ran on", "predicted ms"])
    table["fastest"] = (table["measured ms"] == table["measured ms"].min()).map({True: "<-", False: ""})
    print(f"{op} {size}: AutoHetero ran it on {ran_on(results['AutoHetero'][1])}")
    return table.round(1)

A small image batch: little work per transfer, so the CPU should win.

In [ ]:
compare("sobel", (1, 3, 64, 64))

A large image batch: lots of arithmetic, so the GPU should win by several times.

In [ ]:
compare("sobel", (16, 3, 720, 1280))

Sorting: the CPU wins only on tiny inputs. Sort can't be split, so there is no Hybrid row.

In [ ]:
display(compare("sort", 1000))
compare("sort", 1_000_000)

## Step 4 — live: your own function

Any function becomes dispatchable with one decorator. `splittable=True` promises each output row depends only on its input row, which lets AutoHetero split a batch across both devices. Inputs can be NumPy arrays; the result comes back as NumPy, on the CPU, as passed in.

The first call measures a slice on each device (a trial); after that, the function's own measured times decide.

In [ ]:
@autohetero.auto(splittable=True)
def normalize(x):
    return (x - x.mean(dim=1, keepdim=True)) / x.std(dim=1, keepdim=True)


data = np.random.default_rng(0).random((200_000, 64), dtype=np.float32)
for call in (1, 2, 3):
    start = time.perf_counter()
    out = normalize(data)
    print(f"call {call}: {(time.perf_counter() - start) * 1e3:6.1f} ms on {ran_on(history.last())}")
print("result:", type(out).__name__, out.shape)

## Step 5 — live: a second job on the GPU

`bench/stress.py` starts another process that keeps the GPU busy, like a second user on a shared machine. AutoHetero times each chunk of a long call; when a chunk takes more than 1.5× its prediction, it moves the rest of the call to Hybrid. Watch the "ran on" column change between the two tables.

In [ ]:
from bench import stress

if devices.GPU is None:
    print("No NVIDIA GPU: skipped.")
else:
    display(compare("resnet18", (128, 3, 224, 224)))
    with stress.gpu(device=str(devices.GPU)):
        display(compare("resnet18", (128, 3, 224, 224)))

## Step 6 — what ran where

`autohetero.report()` lists every dispatched function this session: how often each mode was chosen, total time, fallbacks to the CPU after a GPU error, and switches to Hybrid after a slowdown. (The counts include the forced runs above.)

In [ ]:
pd.DataFrame(autohetero.report())

## Step 7 — the measured results (fixed machine)

From the full runs on the RTX 3050 Laptop + Ryzen 7 5800H: 5 timed runs per option after a warm-up, strategies alternating, AutoHetero's own overhead inside every number. `eval/results.ipynb` walks through them in detail.

**RQ1 — can a small learned model predict the faster device?**

In [ ]:
pd.read_csv(ROOT / "eval" / "out" / "rq1.csv")

**RQ2 — does monitoring and rebalancing beat fixed strategies?** Speedup above 1 means AutoHetero was faster.

In [ ]:
display(Image(ROOT / "eval" / "out" / "rq2.png"))
pd.read_csv(ROOT / "eval" / "out" / "rq2_summary.csv")[["condition", "workloads", "speedup_vs_gpu_geomean", "speedup_vs_cpu_geomean", "within_5pct_of_oracle", "switch_rate"]].round(2)

**Hybrid — is splitting one call across both devices worth it?** Gain above 0 means Hybrid beat GPU-only.

In [ ]:
pd.read_csv(ROOT / "eval" / "out" / "hybrid_gain_summary.csv").round(2)

**RQ3 — zero configuration, low overhead?**

In [ ]:
pd.read_csv(ROOT / "eval" / "out" / "rq3.csv")

**What the results say**

| question | answer on this laptop |
|---|---|
| RQ1: predict the faster device? | **Yes on ops it has seen** (91%), 81% from the 33 s calibration alone; **69% on an op it never saw** (fft worst) |
| RQ2: recover time by rebalancing? | Always faster than always-CPU (1.9–3.3×). Slower than always-GPU: 0.70× idle, 0.82× under GPU stress, 0.21× under CPU stress (sort 10M left out: its GPU baseline was inflated by run order) |
| Hybrid gain of 10–30%? | **No**: +11% on one op, most ops lose |
| RQ3: zero configuration, < 1 ms? | **Yes**: ~0.1 ms per call, 1 added line, 33 s one-off calibration |

## Step 8 — limitations and next steps

- **Why slower than always-GPU at idle:** long calls run in chunks so a slowdown can be caught, and each chunk pays its own start-up cost; at idle there is nothing to catch. The tuning constants are untuned first guesses.
- **Why CPU stress goes wrong:** the slowdown check sees a call running slow, but not which device is loaded, so it moved work onto the busy CPU. Next: make the check load-aware.
- **This run:** a 4 GB GPU (the largest image batches didn't fit), Windows, CPU stress by Python busy loops.
- **The M.Tech project** builds on these results: a confidence score on every prediction (RQ1's 69% on unseen ops), measuring only when unsure (the chunking cost at idle), and choosing how many jobs share the GPU (the slowdown from a second GPU job). See `CARRYOVER.md`.